# 03 · Analysis, figures and dashboard  (CPU is enough)
Run after all result files are on GitHub (at least `seed1`-`seed3`; ablations are used if present).
Works on Kaggle (Accelerator = None, Internet = On) or on your laptop (`pip install -r requirements.txt`, then run from the repo folder). About 5 min.

**Output:** `outputs_03.zip` with `tables/` (all CSV tables), `figures/` (12 PNG figures) and `dashboard/dashboard.html`.
Unzip into the repo folder and commit; then send the zip so the report can be written.

In [ ]:
# --- Get the project code from GitHub (works for public repos; for a private repo add a Kaggle
#     secret called GITHUB_TOKEN: Add-ons > Secrets) ---
import os, subprocess, sys
from pathlib import Path
REPO_URL = "https://github.com/DawarAhmed27/DAV-Reseach-DataContamination.git"
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
REPO = WORK / "repo"
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass
if Path("dcl").exists():
    REPO = Path.cwd()          # running locally inside the repo folder
elif REPO.exists():   # update to the latest GitHub version (local generated files are replaced by GitHub's copy)
    subprocess.run(["git", "-C", str(REPO), "fetch", "--quiet"], check=False)
    subprocess.run(["git", "-C", str(REPO), "reset", "--quiet", "--hard", "@{u}"], check=False)
else:
    subprocess.run(["git", "clone", "--quiet", "--depth", "1", url, str(REPO)], check=True)
sys.path.insert(0, str(REPO))
print("Code ready in", REPO)

In [ ]:
!pip install -q plotly statsmodels 2>/dev/null | tail -1
from dcl import analysis as A, figures as F, dashboard as DB
res = A.run_all(REPO)
print("Runs found:", sorted(res["runs"]))
res["cell"][["group", "n_items", "infl_p", "infl_p_lo", "infl_p_hi", "infl_acc_pts"]].round(3)

## RQ2 - detection (AUROC per cell)

In [ ]:
res["detection"].pivot_table(index="detector", columns="group", values="auroc").round(3)

## RQ3 - does the detector score predict inflation?

In [ ]:
display(res["agreement"][res["agreement"]["scope"] != "within cell"].round(3))
display(res["agreement"].groupby("detector")["mean_within_cell_rho"].first().round(3))
display(res["partial"].round(3))
res["predictor"].round(3)

## Statistics, ablations, control, literature comparison

In [ ]:
display(res["mixed"].round(4))
display(res["mcnemar"].round(4))
display(res["ablation"].round(3).T)
display(res["detector_settings"].round(3))
display(res["control"].round(3))
res["literature"]

In [ ]:
paths = F.make_all(res, REPO)
dash = DB.build(res, REPO / "dashboard" / "dashboard.html")
import zipfile
z = WORK / "outputs_03.zip"
with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in ["tables", "figures", "dashboard"]:
        for p in (REPO / folder).glob("*"):
            if p.is_file() and p.name != ".gitkeep":
                zf.write(p, p.relative_to(REPO))
print(len(paths), "figures;", dash, "; download", z)

In [ ]:
from IPython.display import Image, display
for p in paths:
    display(Image(filename=str(p), width=900))